<a href="https://colab.research.google.com/github/WesalFadhl/Murshid-Wesal-Mashael/blob/main/Homework1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
print("Python:", sys.version)

%pip install transformers==4.49.0 sentencepiece==0.2.0

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
  Using cached transformers-4.49.0-py3-none-any.whl.metadata (44 kB)
  Using cached sentencepiece-0.2.0.tar.gz (2.6 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.49.0-py3-none-any.whl (10.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  error: subprocess-exited-with-error
  
  × Building wheel for sentencepiece (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel f

In [2]:
import sys
import subprocess
from pathlib import Path

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.49.0",
    "sentencepiece==0.2.1"
])

import torch
import transformers
import sentencepiece

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("SentencePiece:", sentencepiece.__version__)
print("Setup completed!")

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Torch: 2.14.1+cpu
Transformers: 4.49.0
SentencePiece: 0.2.1
Setup completed!


In [3]:
import time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

start_load = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

model.to("cpu")
model.eval()

load_time = time.perf_counter() - start_load

print("Model:", MODEL_NAME)
print("Device: cpu")
print("Model loaded successfully!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model: google/flan-t5-small
Device: cpu
Model loaded successfully!


In [4]:
CONTEXT = (
    "You can check your application status "
    "using its reference number."
)

QUESTION = "How can I check my application status?"

prompt = f"""
Answer the question using only the context.

Context: {CONTEXT}

Question: {QUESTION}

Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt")

start = time.perf_counter()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

inference_time = time.perf_counter() - start

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Context:", CONTEXT)
print("Question:", QUESTION)
print("Generated answer:", answer)

Context: You can check your application status using its reference number.
Question: How can I check my application status?
Generated answer: Use its reference number


In [5]:
import json
from pathlib import Path

OUTPUT_DIR = Path("murshid_live_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

report = {
    "mode": "real_local_model",
    "model": MODEL_NAME,
    "device": "cpu",
    "context": CONTEXT,
    "question": QUESTION,
    "answer": answer,
    "model_load_time_seconds": round(load_time, 4),
    "inference_time_seconds": round(inference_time, 4),
    "input_token_count": len(inputs["input_ids"][0]),
    "output_token_count": len(outputs[0]),
    "external_inference_api_calls": 0
}

output_path = OUTPUT_DIR / "live_demo_result.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))
print("\nReport saved to:", output_path)

{
  "mode": "real_local_model",
  "model": "google/flan-t5-small",
  "device": "cpu",
  "context": "You can check your application status using its reference number.",
  "question": "How can I check my application status?",
  "answer": "Use its reference number",
  "model_load_time_seconds": 10.9066,
  "inference_time_seconds": 0.544,
  "input_token_count": 36,
  "output_token_count": 6,
  "external_inference_api_calls": 0
}

Report saved to: murshid_live_outputs/live_demo_result.json


The input question and context are converted into tokens by the tokenizer and passed to the FLAN-T5 model, which generates an answer token by token.

Unlike FakeClient, which returns predefined answers for specific inputs, the real model generates responses using its learned parameters rather than fixed response mappings.